# HWM Composite Sensitivity

**Purpose.** Exploratory sensitivity check for adding `hwm_heatwave_magnitude` alongside `hwa_heatwave_amplitude` in the Heat Risk bundle before any production methodology change lands.

**Scope.** District level, `historical / 1990-2010 / mean`. Baseline reconstruction is scored per state to match the persisted composite builder. The national-pooled view is intentionally secondary and caveated because production Heat Risk composites are normalized within each state.

**Read/write contract.** Reads existing processed masters only. Writes small exploratory outputs under `scratch/results/hwm_composite_sensitivity/`; it does not write to `IRT_DATA_DIR`.

**Important caveat.** `hwm` does not exist on disk yet. The three scenarios below proxy its normalized behavior, so results are a sensitivity band rather than a prediction.

## 1. Configuration

In [2]:
from __future__ import annotations

import json
import os
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd


import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "india_resilience_tool").exists():
    raise RuntimeError(f"Could not locate repo root from cwd={Path.cwd()}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


from india_resilience_tool.analysis.bundle_scores import (
    BundleMetricSpec,
    compute_bundle_score_frame,
    normalized_metric_column,
)
from india_resilience_tool.compute.composite_metrics import (
    LEGACY_MASTER_FILENAMES,
    SUPPORTED_STAT,
    _build_wide_component_frame,
    _bundle_metric_specs,
    _discover_states_for_spec,
    _load_component_master,
    _required_id_columns,
)
from india_resilience_tool.config.composite_metrics import COMPOSITES_BY_BUNDLE
from india_resilience_tool.config.metrics_registry import METRICS_BY_SLUG
from india_resilience_tool.config.paths import get_paths_config, resolve_processed_root
from india_resilience_tool.data.master_loader import normalize_master_columns

BUNDLE = "Heat Risk"
LEVEL = "district"
REQUESTED_SCENARIO = "historical"
REQUESTED_PERIOD = "1990-2010"
FALLBACK_SCENARIO = "ssp585"
FALLBACK_PERIOD = "2040-2060"
STAT = "mean"
SEED = 20260715

HWA = "hwa_heatwave_amplitude"
HWM = "hwm_heatwave_magnitude"
TAS = "tas_annual_mean"
TXX = "txx_annual_max"
TN90P = "tn90p_warm_nights_pct"
EXTREMES = {TXX, TN90P, HWA}

paths = get_paths_config()
DATA_DIR = Path(os.getenv("IRT_DATA_DIR", paths.data_dir)).expanduser().resolve()
OUT_DIR = Path("scratch/results/hwm_composite_sensitivity")
OUT_DIR.mkdir(parents=True, exist_ok=True)

spec = COMPOSITES_BY_BUNDLE[BUNDLE]
id_columns = list(_required_id_columns(LEVEL))

def _score_column_for(scenario: str, period: str) -> str:
    return f"{spec.composite_slug}__{scenario}__{period}__{STAT}"


def _persisted_composite_columns() -> set[str]:
    root = resolve_processed_root(spec.composite_slug, data_dir=DATA_DIR, mode="portfolio")
    columns: set[str] = set()
    for path in root.glob(f"*/{LEGACY_MASTER_FILENAMES[LEVEL]}"):
        parquet = path.with_suffix(".parquet")
        if parquet.exists():
            frame = pd.read_parquet(parquet)
            columns.update(str(c) for c in frame.columns if str(c).startswith(f"{spec.composite_slug}__"))
    return columns


persisted_score_columns = _persisted_composite_columns()
requested_score_col = _score_column_for(REQUESTED_SCENARIO, REQUESTED_PERIOD)
fallback_score_col = _score_column_for(FALLBACK_SCENARIO, FALLBACK_PERIOD)
if requested_score_col in persisted_score_columns:
    SCENARIO = REQUESTED_SCENARIO
    PERIOD = REQUESTED_PERIOD
elif fallback_score_col in persisted_score_columns:
    SCENARIO = FALLBACK_SCENARIO
    PERIOD = FALLBACK_PERIOD
else:
    if not persisted_score_columns:
        raise FileNotFoundError("No persisted Heat Risk composite score columns found for fidelity check.")
    first = sorted(persisted_score_columns)[0]
    _, SCENARIO, PERIOD, _ = first.split("__", 3)
score_col = _score_column_for(SCENARIO, PERIOD)

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR : {OUT_DIR.resolve()}")
print(f"Composite: {spec.composite_slug}")
print(f"Requested slice: {REQUESTED_SCENARIO} / {REQUESTED_PERIOD} / {STAT}")
print(f"Effective slice : {SCENARIO} / {PERIOD} / {STAT}")
if (SCENARIO, PERIOD) != (REQUESTED_SCENARIO, REQUESTED_PERIOD):
    print("NOTE: Requested historical slice is not present in persisted composite parquet; using a persisted production slice for the fidelity-backed run.")
print(f"Baseline score column: {score_col}")

DATA_DIR: D:\projects\irt_data
OUT_DIR : D:\projects\india_resilience_tool\notebooks\scratch\results\hwm_composite_sensitivity
Composite: composite_heat_risk
Requested slice: historical / 1990-2010 / mean
Effective slice : ssp585 / 2040-2060 / mean
NOTE: Requested historical slice is not present in persisted composite parquet; using a persisted production slice for the fidelity-backed run.
Baseline score column: composite_heat_risk__ssp585__2040-2060__mean


## 2. Helpers

In [3]:
def _component_frames_for_state(state_name: str) -> dict[str, pd.DataFrame]:
    frames: dict[str, pd.DataFrame] = {}
    for metric_slug in spec.component_metric_slugs:
        frame = _load_component_master(metric_slug, level=LEVEL, state_name=state_name, data_dir=DATA_DIR)
        if frame is None or frame.empty:
            return {}
        frames[metric_slug] = frame
    return frames


def _wide_for_state(state_name: str) -> pd.DataFrame:
    frames = _component_frames_for_state(state_name)
    if not frames:
        return pd.DataFrame(columns=id_columns + list(spec.component_metric_slugs))
    return _build_wide_component_frame(frames, level=LEVEL, scenario=SCENARIO, period=PERIOD)


def _score_wide(wide: pd.DataFrame, metric_specs: list[BundleMetricSpec]) -> pd.DataFrame:
    return compute_bundle_score_frame(wide, metric_specs=metric_specs, id_columns=id_columns)


def _load_persisted_composite(state_name: str) -> pd.DataFrame:
    root = resolve_processed_root(spec.composite_slug, data_dir=DATA_DIR, mode="portfolio")
    path = root / state_name / LEGACY_MASTER_FILENAMES[LEVEL]
    parquet = path.with_suffix(".parquet")
    if not parquet.exists():
        raise FileNotFoundError(f"Missing persisted composite parquet: {parquet}")
    return normalize_master_columns(pd.read_parquet(parquet))


def _modified_metric_specs() -> list[BundleMetricSpec]:
    modified: list[BundleMetricSpec] = []
    for base in _bundle_metric_specs(spec):
        if base.slug in EXTREMES:
            modified.append(replace(base, weight=0.25 / 4.0))
        else:
            modified.append(base)
    modified.append(
        BundleMetricSpec(
            slug=HWM,
            label="Heatwave Magnitude (proxied mean exceedance)",
            column=HWM,
            weight=0.25 / 4.0,
            higher_is_worse=True,
        )
    )
    return modified


def _inject_hwm_raw_from_norm(wide: pd.DataFrame, norm_values: pd.Series) -> pd.DataFrame:
    # compute_bundle_score_frame will min-max normalize this synthetic raw column.
    out = wide.copy()
    out[HWM] = pd.to_numeric(norm_values, errors="coerce")
    return out


def _scenario_hwm_norm(wide: pd.DataFrame, baseline_score: pd.DataFrame, scenario_name: str, rng: np.random.Generator) -> pd.Series:
    if scenario_name == "S1_correlated_hwa":
        return baseline_score[normalized_metric_column(HWA)]
    if scenario_name == "S2_cold_favoring_tas_inverse":
        tas_norm = baseline_score[normalized_metric_column(TAS)]
        return 100.0 - pd.to_numeric(tas_norm, errors="coerce")
    if scenario_name == "S3_shuffled_hwa":
        source = baseline_score[normalized_metric_column(HWA)].to_numpy(dtype=float, copy=True)
        finite_idx = np.flatnonzero(np.isfinite(source))
        shuffled = source.copy()
        shuffled[finite_idx] = rng.permutation(source[finite_idx])
        return pd.Series(shuffled, index=baseline_score.index)
    raise ValueError(f"Unknown scenario: {scenario_name}")


def _rank_deltas(frame: pd.DataFrame, score_column: str, modified_column: str, group_column: str | None) -> pd.DataFrame:
    out = frame.copy()
    if group_column:
        out["baseline_rank"] = out.groupby(group_column)[score_column].rank(ascending=False, method="min")
        out["modified_rank"] = out.groupby(group_column)[modified_column].rank(ascending=False, method="min")
    else:
        out["baseline_rank"] = out[score_column].rank(ascending=False, method="min")
        out["modified_rank"] = out[modified_column].rank(ascending=False, method="min")
    out["rank_shift"] = out["modified_rank"] - out["baseline_rank"]
    out["abs_rank_shift"] = out["rank_shift"].abs()
    return out


def _corr(a: pd.Series, b: pd.Series, method: str) -> float:
    valid = pd.concat([a, b], axis=1).dropna()
    if len(valid) < 3:
        return float("nan")
    return float(valid.iloc[:, 0].corr(valid.iloc[:, 1], method=method))


def _distribution_summary(values: pd.Series) -> dict[str, float]:
    x = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if x.empty:
        return {"n": 0, "mean": np.nan, "std": np.nan, "p05": np.nan, "median": np.nan, "p95": np.nan, "min": np.nan, "max": np.nan}
    return {
        "n": int(x.size),
        "mean": float(x.mean()),
        "std": float(x.std(ddof=1)) if x.size > 1 else 0.0,
        "p05": float(x.quantile(0.05)),
        "median": float(x.median()),
        "p95": float(x.quantile(0.95)),
        "min": float(x.min()),
        "max": float(x.max()),
    }

## 3. Reconstruct Baseline And Run Fidelity Gate

In [4]:
states = _discover_states_for_spec(spec, level=LEVEL, data_dir=DATA_DIR)
print(f"Discovered states: {len(states)}")

baseline_specs = _bundle_metric_specs(spec)
wide_by_state: dict[str, pd.DataFrame] = {}
baseline_parts: list[pd.DataFrame] = []
persisted_parts: list[pd.DataFrame] = []

for state_name in states:
    wide = _wide_for_state(state_name)
    if wide.empty:
        continue
    wide_by_state[state_name] = wide
    scored = _score_wide(wide, baseline_specs)
    baseline_parts.append(scored.assign(state_partition=state_name))

    persisted = _load_persisted_composite(state_name)
    persisted_parts.append(persisted[id_columns + [score_col]].assign(state_partition=state_name))

baseline_recon = pd.concat(baseline_parts, ignore_index=True)
persisted = pd.concat(persisted_parts, ignore_index=True)

fidelity = baseline_recon[id_columns + ["bundle_score"]].merge(
    persisted[id_columns + [score_col]], on=id_columns, how="outer", validate="one_to_one"
)
delta = pd.to_numeric(fidelity["bundle_score"], errors="coerce") - pd.to_numeric(fidelity[score_col], errors="coerce")
nan_aligned = fidelity["bundle_score"].isna().equals(fidelity[score_col].isna())
max_abs_diff = float(delta.abs().max(skipna=True)) if delta.notna().any() else 0.0

print(f"Baseline rows: {len(baseline_recon):,}")
print(f"States scored: {len(wide_by_state):,}")
print(f"NaN aligned: {nan_aligned}")
print(f"Max abs diff vs persisted parquet: {max_abs_diff:.12g}")

assert nan_aligned, "Baseline reconstruction NaNs do not align with persisted composite parquet. Check id-key join, directionality, renorm divisor, or stale on-disk composite."
assert max_abs_diff < 1e-6, "Baseline reconstruction failed fidelity gate. Check id-key join, directionality, renorm divisor, or stale on-disk composite parquet."
assert not np.isinf(baseline_recon.select_dtypes(include=[np.number])).any().any(), "Baseline reconstruction contains infinite values."

baseline_recon.head()

Discovered states: 36
Baseline rows: 781
States scored: 35
NaN aligned: True
Max abs diff vs persisted parquet: 0


,state,district,district_key,tas_annual_mean__landing_norm,tasmax_summer_mean__landing_norm,tas_summer_mean__landing_norm,txx_annual_max__landing_norm,tn90p_warm_nights_pct__landing_norm,hwa_heatwave_amplitude__landing_norm,txge30_hot_days__landing_norm,txge35_extreme_heat_days__landing_norm,tasmin_tropical_nights_gt25__landing_norm,hwfi_tmean_90p__landing_norm,hwfi_events_tmean_90p__landing_norm,wsdi_warm_spell_days__landing_norm,tnx_annual_max__landing_norm,tx90p_hot_days_pct__landing_norm,bundle_score,available_metric_count,state_partition
0,Andaman & Nicobar Islands,Nicobars,andaman nicobar islands|nicobars,97.791192,56.501692,72.873542,35.175760,100.000000,55.481619,81.99645,50.000000,56.434455,100.000000,0.000000,100.000000,61.482061,100.000000,68.526741,14,Andaman & Nicobar Islands
1,Andaman & Nicobar Islands,North and Middle Andaman,andaman nicobar islands|north and middle andaman,100.000000,100.000000,100.000000,100.000000,0.000000,100.000000,100.00000,50.000000,100.000000,0.000000,100.000000,0.000000,100.000000,0.000000,67.500000,14,Andaman & Nicobar Islands
2,Andaman & Nicobar Islands,South Andamans,andaman nicobar islands|south andamans,0.000000,0.000000,0.000000,0.000000,21.179715,0.000000,0.00000,50.000000,0.000000,23.893527,63.143815,29.423443,0.000000,29.117598,15.528846,14,Andaman & Nicobar Islands
3,Andhra Pradesh,Alluri Sitharama Raju,andhra pradesh|alluri sitharama raju,0.000000,17.191824,0.000000,33.243432,53.756530,21.197400,0.00000,10.906582,10.737123,43.691186,28.239646,4.890377,12.649020,13.404671,19.063232,14,Andhra Pradesh
4,Andhra Pradesh,Anakapalli,andhra pradesh|anakapalli,54.743541,31.504154,37.488435,38.190068,83.907535,37.275648,75.13220,36.266518,81.120101,72.561110,66.101005,44.199202,51.100370,53.358171,54.674942,14,Andhra Pradesh


## 4. Per-State Sensitivity Scenarios

In [5]:
scenario_names = [
    "S1_correlated_hwa",
    "S2_cold_favoring_tas_inverse",
    "S3_shuffled_hwa",
]
modified_specs = _modified_metric_specs()

scenario_frames: dict[str, pd.DataFrame] = {}
rng = np.random.default_rng(SEED)

for scenario_name in scenario_names:
    parts: list[pd.DataFrame] = []
    for state_name, wide in wide_by_state.items():
        baseline_state = baseline_recon.loc[baseline_recon["state_partition"] == state_name].reset_index(drop=True)
        wide_state = wide.reset_index(drop=True)
        hwm_norm = _scenario_hwm_norm(wide_state, baseline_state, scenario_name, rng)
        modified_wide = _inject_hwm_raw_from_norm(wide_state, hwm_norm)
        modified_score = _score_wide(modified_wide, modified_specs).rename(
            columns={"bundle_score": "modified_score", "available_metric_count": "modified_available_metric_count"}
        )
        base_cols = baseline_state[id_columns + ["bundle_score", "available_metric_count"]].rename(
            columns={"bundle_score": "baseline_score", "available_metric_count": "baseline_available_metric_count"}
        )
        joined = base_cols.merge(
            modified_score[id_columns + ["modified_score", "modified_available_metric_count"]],
            on=id_columns,
            how="inner",
            validate="one_to_one",
        )
        joined["scenario"] = scenario_name
        joined["state_partition"] = state_name
        joined["score_delta"] = joined["modified_score"] - joined["baseline_score"]
        joined["available_metric_count_delta"] = joined["modified_available_metric_count"] - joined["baseline_available_metric_count"]
        parts.append(joined)
    frame = pd.concat(parts, ignore_index=True)
    frame = _rank_deltas(frame, "baseline_score", "modified_score", "state")
    scenario_frames[scenario_name] = frame

per_state_results = pd.concat(scenario_frames.values(), ignore_index=True)
assert not np.isinf(per_state_results.select_dtypes(include=[np.number])).any().any(), "Per-state results contain infinite values."

per_state_results.groupby("scenario")["score_delta"].agg(["count", "mean", "std", "min", "median", "max"])

,count,mean,std,min,median,max
scenario,,,,,,
S1_correlated_hwa,781,0.296377,1.015515,-2.175578,0.314369,2.662696
S2_cold_favoring_tas_inverse,781,-0.991788,2.559608,-6.250000,-1.417296,6.250000
S3_shuffled_hwa,781,0.296377,1.977072,-5.641815,0.470798,6.181406


## 5. Within-State Summary

In [6]:
state_sizes = baseline_recon.groupby("state", dropna=False).size().rename("n_districts").reset_index()
small_states = state_sizes.loc[state_sizes["n_districts"] < 3, "state"].tolist()
print(f"States with <3 districts: {len(small_states)}")
if small_states:
    print(", ".join(map(str, small_states)))

rows: list[dict[str, object]] = []
state_corr_rows: list[dict[str, object]] = []
for scenario_name, frame in scenario_frames.items():
    dist = _distribution_summary(frame["score_delta"])
    rows.append({
        "view": "within_state",
        "scenario": scenario_name,
        **{f"delta_{k}": v for k, v in dist.items()},
        "median_abs_rank_shift": float(frame["abs_rank_shift"].median()),
        "p95_abs_rank_shift": float(frame["abs_rank_shift"].quantile(0.95)),
        "moved_gt_5": int((frame["abs_rank_shift"] > 5).sum()),
        "moved_gt_10": int((frame["abs_rank_shift"] > 10).sum()),
        "district_rows": int(len(frame)),
        "seed": SEED if scenario_name == "S3_shuffled_hwa" else np.nan,
    })
    for state_name, g in frame.groupby("state", dropna=False):
        if len(g) < 3:
            continue
        state_corr_rows.append({
            "scenario": scenario_name,
            "state": state_name,
            "n_districts": int(len(g)),
            "spearman": _corr(g["baseline_score"], g["modified_score"], "spearman"),
            "kendall": _corr(g["baseline_score"], g["modified_score"], "kendall"),
            "median_abs_rank_shift": float(g["abs_rank_shift"].median()),
            "p95_abs_rank_shift": float(g["abs_rank_shift"].quantile(0.95)),
        })

state_corr = pd.DataFrame(state_corr_rows)
corr_summary = state_corr.groupby("scenario").agg(
    states=("state", "count"),
    median_spearman=("spearman", "median"),
    median_kendall=("kendall", "median"),
    median_state_p95_abs_rank_shift=("p95_abs_rank_shift", "median"),
).reset_index()

summary = pd.DataFrame(rows).merge(corr_summary, on="scenario", how="left")
summary

States with <3 districts: 4
Chandigarh, Goa, Ladakh, Lakshadweep


,view,scenario,delta_n,delta_mean,delta_std,delta_p05,delta_median,delta_p95,delta_min,delta_max,median_abs_rank_shift,p95_abs_rank_shift,moved_gt_5,moved_gt_10,district_rows,seed,states,median_spearman,median_kendall,median_state_p95_abs_rank_shift
0,within_state,S1_correlated_hwa,781,0.296377,1.015515,-1.363803,0.314369,1.841544,-2.175578,2.662696,0.0,3.0,7,0,781,NaN,31,0.994216,0.960474,1.00
1,within_state,S2_cold_favoring_tas_inverse,781,-0.991788,2.559608,-4.387648,-1.417296,4.166667,-6.250000,6.250000,0.0,3.0,16,2,781,NaN,31,0.990196,0.948052,1.75
2,within_state,S3_shuffled_hwa,781,0.296377,1.977072,-3.394957,0.470798,3.347704,-5.641815,6.181406,1.0,5.0,39,11,781,20260715.0,31,0.981931,0.913978,2.90


## 6. Caveated National-Pooled Pass

In [7]:
national_wide = pd.concat(wide_by_state.values(), ignore_index=True)
national_baseline = _score_wide(national_wide, baseline_specs).rename(
    columns={"bundle_score": "baseline_score", "available_metric_count": "baseline_available_metric_count"}
)

national_frames: dict[str, pd.DataFrame] = {}
national_rng = np.random.default_rng(SEED)
for scenario_name in scenario_names:
    hwm_norm = _scenario_hwm_norm(national_wide, national_baseline, scenario_name, national_rng)
    modified_wide = _inject_hwm_raw_from_norm(national_wide, hwm_norm)
    modified = _score_wide(modified_wide, modified_specs).rename(
        columns={"bundle_score": "modified_score", "available_metric_count": "modified_available_metric_count"}
    )
    frame = national_baseline[id_columns + ["baseline_score", "baseline_available_metric_count"]].merge(
        modified[id_columns + ["modified_score", "modified_available_metric_count"]],
        on=id_columns,
        how="inner",
        validate="one_to_one",
    )
    frame["scenario"] = scenario_name
    frame["score_delta"] = frame["modified_score"] - frame["baseline_score"]
    frame = _rank_deltas(frame, "baseline_score", "modified_score", None)
    national_frames[scenario_name] = frame

national_results = pd.concat(national_frames.values(), ignore_index=True)
national_rows = []
for scenario_name, frame in national_frames.items():
    dist = _distribution_summary(frame["score_delta"])
    national_rows.append({
        "view": "national_pooled_caveated",
        "scenario": scenario_name,
        **{f"delta_{k}": v for k, v in dist.items()},
        "spearman": _corr(frame["baseline_score"], frame["modified_score"], "spearman"),
        "kendall": _corr(frame["baseline_score"], frame["modified_score"], "kendall"),
        "median_abs_rank_shift": float(frame["abs_rank_shift"].median()),
        "p95_abs_rank_shift": float(frame["abs_rank_shift"].quantile(0.95)),
        "moved_gt_5": int((frame["abs_rank_shift"] > 5).sum()),
        "moved_gt_10": int((frame["abs_rank_shift"] > 10).sum()),
        "district_rows": int(len(frame)),
        "seed": SEED if scenario_name == "S3_shuffled_hwa" else np.nan,
    })
national_summary = pd.DataFrame(national_rows)
national_summary

,view,scenario,delta_n,delta_mean,delta_std,delta_p05,delta_median,delta_p95,delta_min,delta_max,spearman,kendall,median_abs_rank_shift,p95_abs_rank_shift,moved_gt_5,moved_gt_10,district_rows,seed
0,national_pooled_caveated,S1_correlated_hwa,781,1.232209,0.452614,0.372198,1.382558,1.742241,-0.564315,1.859816,0.998304,0.971135,6.0,24.0,403,237,781,NaN
1,national_pooled_caveated,S2_cold_favoring_tas_inverse,781,-2.920065,1.572683,-4.155555,-3.408005,0.772850,-4.359653,5.718109,0.998569,0.971056,7.0,24.0,436,262,781,NaN
2,national_pooled_caveated,S3_shuffled_hwa,781,1.232209,1.235372,-1.078941,1.393976,2.985961,-4.343104,5.410927,0.975775,0.890673,20.0,97.0,607,510,781,20260715.0


## 7. Movers And Cold/Hot Contrast

In [8]:
def _top_movers(frame: pd.DataFrame, n: int = 12) -> pd.DataFrame:
    cols = ["state", "district", "baseline_score", "modified_score", "score_delta", "baseline_rank", "modified_rank", "rank_shift", "abs_rank_shift"]
    return frame.sort_values(["abs_rank_shift", "score_delta"], ascending=[False, False]).loc[:, cols].head(n)


for scenario_name in scenario_names:
    print(f"\n=== {scenario_name}: within-state movers ===")
    display(_top_movers(scenario_frames[scenario_name]))
    print(f"\n=== {scenario_name}: national-pooled movers (caveated) ===")
    display(_top_movers(national_frames[scenario_name]))

contrast_names = ["Lahul", "Spiti", "Leh", "Mangan", "Kinnaur", "Tawang", "Jalgaon", "Anand", "Kheda"]
contrast_pattern = "|".join(contrast_names)
contrast = national_results.loc[
    national_results["district"].astype(str).str.contains(contrast_pattern, case=False, na=False),
    ["scenario", "state", "district", "baseline_score", "modified_score", "score_delta", "baseline_rank", "modified_rank", "rank_shift"],
].sort_values(["scenario", "district"])
contrast


=== S1_correlated_hwa: within-state movers ===


,state,district,baseline_score,modified_score,score_delta,baseline_rank,modified_rank,rank_shift,abs_rank_shift
725,Uttar Pradesh,Mirzapur,61.229976,62.508971,1.278995,33.0,23.0,-10.0,10.0
719,Uttar Pradesh,Maharajganj,63.806419,62.292911,-1.513508,15.0,24.0,9.0,9.0
738,Uttar Pradesh,Shravasti,63.639911,62.285695,-1.354216,17.0,25.0,8.0,8.0
404,Maharashtra,Mumbai,52.663503,52.022837,-0.640667,14.0,21.0,7.0,7.0
690,Uttar Pradesh,Chandauli,62.205530,63.185209,0.979679,26.0,20.0,-6.0,6.0
716,Uttar Pradesh,Kushi Nagar,62.615580,61.356558,-1.259023,23.0,29.0,6.0,6.0
739,Uttar Pradesh,Siddharth Nagar,61.854761,60.560288,-1.294473,27.0,33.0,6.0,6.0
97,Bihar,Buxar,65.502404,66.857076,1.354671,12.0,7.0,-5.0,5.0
711,Uttar Pradesh,Kanpur Dehat,62.482506,63.394037,0.911530,24.0,19.0,-5.0,5.0
675,Uttar Pradesh,Auraiya,59.852802,60.340384,0.487582,39.0,34.0,-5.0,5.0



=== S1_correlated_hwa: national-pooled movers (caveated) ===


,state,district,baseline_score,modified_score,score_delta,baseline_rank,modified_rank,rank_shift,abs_rank_shift
2,Andaman & Nicobar Islands,South Andamans,58.616490,58.429263,-0.187227,295.0,401.0,106.0,106.0
335,Lakshadweep,Lakshadweep,61.283728,60.719413,-0.564315,147.0,241.0,94.0,94.0
329,Kerala,Pathanamthitta,59.768702,60.076472,0.307770,205.0,295.0,90.0,90.0
1,Andaman & Nicobar Islands,North and Middle Andaman,62.299570,62.266807,-0.032764,108.0,167.0,59.0,59.0
0,Andaman & Nicobar Islands,Nicobars,64.247005,63.789569,-0.457436,46.0,105.0,59.0,59.0
601,Tamil Nadu,Kanniyakumari,61.224885,61.266768,0.041883,148.0,206.0,58.0,58.0
328,Kerala,Palakkad,57.921773,58.639360,0.717587,337.0,392.0,55.0,55.0
315,Karnataka,Uttara Kannada,58.714026,59.446594,0.732568,286.0,334.0,48.0,48.0
402,Maharashtra,Kolhapur,57.584593,58.382262,0.797669,360.0,405.0,45.0,45.0
594,Tamil Nadu,Coimbatore,56.176186,56.988092,0.811907,455.0,499.0,44.0,44.0



=== S2_cold_favoring_tas_inverse: within-state movers ===


...


=== S3_shuffled_hwa: within-state movers ===


,state,district,baseline_score,modified_score,score_delta,baseline_rank,modified_rank,rank_shift,abs_rank_shift
702,Uttar Pradesh,Gorakhpur,60.549897,62.345419,1.795523,37.0,17.0,-20.0,20.0
731,Uttar Pradesh,Rae Bareli,65.365288,61.646170,-3.719118,10.0,27.0,17.0,17.0
707,Uttar Pradesh,Jalaun,63.412341,66.043688,2.631347,20.0,6.0,-14.0,14.0
743,Uttar Pradesh,Unnao,61.401765,62.374470,0.972705,30.0,16.0,-14.0,14.0
742,Uttar Pradesh,Sultanpur,63.475707,60.944850,-2.530858,19.0,33.0,14.0,14.0
672,Uttar Pradesh,Ambedkar Nagar,61.311021,62.289674,0.978653,32.0,19.0,-13.0,13.0
718,Uttar Pradesh,Lucknow,60.186193,61.781128,1.594935,38.0,26.0,-12.0,12.0
681,Uttar Pradesh,Balrampur,62.242770,63.218362,0.975591,25.0,13.0,-12.0,12.0
711,Uttar Pradesh,Kanpur Dehat,62.482506,60.073549,-2.408957,24.0,36.0,12.0,12.0
690,Uttar Pradesh,Chandauli,62.205530,59.060711,-3.144819,26.0,38.0,12.0,12.0



=== S3_shuffled_hwa: national-pooled movers (caveated) ===


,state,district,baseline_score,modified_score,score_delta,baseline_rank,modified_rank,rank_shift,abs_rank_shift
96,Bihar,Bhojpur,59.330191,55.508301,-3.821890,232.0,552.0,320.0,320.0
18,Andhra Pradesh,Palnadu,62.387682,58.803177,-3.584505,102.0,350.0,248.0,248.0
653,Telangana,Rajanna Sircilla,61.001230,58.030887,-2.970342,156.0,397.0,241.0,241.0
672,Uttar Pradesh,Ambedkar Nagar,58.634483,56.107027,-2.527456,292.0,524.0,232.0,232.0
289,Karnataka,Ballari,61.852013,58.682564,-3.169449,131.0,355.0,224.0,224.0
304,Karnataka,Kalaburagi,62.357040,59.122689,-3.234351,104.0,323.0,219.0,219.0
694,Uttar Pradesh,Etawah,58.226922,55.875049,-2.351873,321.0,538.0,217.0,217.0
416,Maharashtra,Sangli,59.423044,57.533811,-1.889232,221.0,435.0,214.0,214.0
341,Madhya Pradesh,Barwani,57.178540,53.947249,-3.231291,389.0,598.0,209.0,209.0
642,Telangana,Mahabubnagar,59.962295,57.972467,-1.989828,195.0,399.0,204.0,204.0


,scenario,state,district,baseline_score,modified_score,score_delta,baseline_rank,modified_rank,rank_shift
177,S1_correlated_hwa,Gujarat,Anand,64.147754,65.692146,1.544392,48.0,42.0,-6.0
400,S1_correlated_hwa,Maharashtra,Jalgaon,60.800974,62.588964,1.787990,165.0,151.0,-14.0
192,S1_correlated_hwa,Gujarat,Kheda,63.608298,65.216472,1.608174,61.0,52.0,-9.0
234,S1_correlated_hwa,Himachal Pradesh,Kinnaur,22.844555,22.804147,-0.040407,767.0,768.0,1.0
236,S1_correlated_hwa,Himachal Pradesh,Lahul and Spiti,11.698525,11.268244,-0.430281,780.0,780.0,0.0
334,S1_correlated_hwa,Ladakh,Leh Ladakh,8.509776,8.413145,-0.096630,781.0,781.0,0.0
587,S1_correlated_hwa,Sikkim,Mangan,12.873378,12.726288,-0.147090,778.0,778.0,0.0
49,S1_correlated_hwa,Arunachal Pradesh,Tawang,21.370685,21.824083,0.453398,771.0,771.0,0.0
958,S2_cold_favoring_tas_inverse,Gujarat,Anand,64.147754,59.885997,-4.261758,48.0,53.0,5.0
1181,S2_cold_favoring_tas_inverse,Maharashtra,Jalgaon,60.800974,56.665580,-4.135395,165.0,177.0,12.0


## 8. Write Exploratory Artifacts

In [9]:
combined_summary = pd.concat([summary, national_summary], ignore_index=True, sort=False)
summary_path = OUT_DIR / "hwm_composite_sensitivity_summary.csv"
within_path = OUT_DIR / "hwm_composite_sensitivity_within_state_rows.csv"
national_path = OUT_DIR / "hwm_composite_sensitivity_national_pooled_rows.csv"
md_path = OUT_DIR / "hwm_composite_sensitivity_summary.md"

combined_summary.to_csv(summary_path, index=False)
per_state_results.to_csv(within_path, index=False)
national_results.to_csv(national_path, index=False)

def _markdown_table(frame: pd.DataFrame) -> str:
    try:
        return frame.to_markdown(index=False)
    except ImportError:
        return "```text\n" + frame.to_string(index=False) + "\n```"


lines = [
    "# HWM Composite Sensitivity Summary",
    "",
    f"Data dir: `{DATA_DIR}`",
    f"Rows: `{len(baseline_recon):,}` district rows across `{len(wide_by_state)}` states",
    f"Fidelity max abs diff vs persisted parquet: `{max_abs_diff:.12g}`",
    f"NaN aligned: `{nan_aligned}`",
    f"Shuffle seed: `{SEED}`",
    f"States with <3 districts: `{len(small_states)}`",
    "",
    "## Within-State View (Production-Matching Normalization)",
    _markdown_table(summary),
    "",
    "## National-Pooled View (Caveated, Not Production)",
    _markdown_table(national_summary),
    "",
    "## Notes",
    "- `S1_correlated_hwa` is the reweight-only floor because proxied HWM follows normalized HWA.",
    "- `S2_cold_favoring_tas_inverse` is an assumption-heavy anomaly-lens proxy using `100 - norm(tas_annual_mean)` per comparison frame.",
    "- `S3_shuffled_hwa` is a reproducible noise ceiling using the recorded seed.",
    "- The national-pooled pass normalizes raw component columns once across India and is not how production persisted composites are built.",
]
md_path.write_text("\n".join(lines), encoding="utf-8")

manifest = {
    "data_dir": str(DATA_DIR),
    "summary_csv": str(summary_path),
    "within_state_csv": str(within_path),
    "national_pooled_csv": str(national_path),
    "summary_md": str(md_path),
    "scenario": SCENARIO,
    "period": PERIOD,
    "stat": STAT,
    "seed": SEED,
    "baseline_rows": int(len(baseline_recon)),
    "states_scored": int(len(wide_by_state)),
    "max_abs_diff": max_abs_diff,
    "nan_aligned": bool(nan_aligned),
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Wrote {summary_path}")
print(f"Wrote {within_path}")
print(f"Wrote {national_path}")
print(f"Wrote {md_path}")
combined_summary

Wrote scratch\results\hwm_composite_sensitivity\hwm_composite_sensitivity_summary.csv
Wrote scratch\results\hwm_composite_sensitivity\hwm_composite_sensitivity_within_state_rows.csv
Wrote scratch\results\hwm_composite_sensitivity\hwm_composite_sensitivity_national_pooled_rows.csv
Wrote scratch\results\hwm_composite_sensitivity\hwm_composite_sensitivity_summary.md


,view,scenario,delta_n,delta_mean,delta_std,delta_p05,delta_median,delta_p95,delta_min,delta_max,...,moved_gt_5,moved_gt_10,district_rows,seed,states,median_spearman,median_kendall,median_state_p95_abs_rank_shift,spearman,kendall
0,within_state,S1_correlated_hwa,781,0.296377,1.015515,-1.363803,0.314369,1.841544,-2.175578,2.662696,...,7,0,781,NaN,31.0,0.994216,0.960474,1.00,NaN,NaN
1,within_state,S2_cold_favoring_tas_inverse,781,-0.991788,2.559608,-4.387648,-1.417296,4.166667,-6.250000,6.250000,...,16,2,781,NaN,31.0,0.990196,0.948052,1.75,NaN,NaN
2,within_state,S3_shuffled_hwa,781,0.296377,1.977072,-3.394957,0.470798,3.347704,-5.641815,6.181406,...,39,11,781,20260715.0,31.0,0.981931,0.913978,2.90,NaN,NaN
3,national_pooled_caveated,S1_correlated_hwa,781,1.232209,0.452614,0.372198,1.382558,1.742241,-0.564315,1.859816,...,403,237,781,NaN,NaN,NaN,NaN,NaN,0.998304,0.971135
4,national_pooled_caveated,S2_cold_favoring_tas_inverse,781,-2.920065,1.572683,-4.155555,-3.408005,0.772850,-4.359653,5.718109,...,436,262,781,NaN,NaN,NaN,NaN,NaN,0.998569,0.971056
5,national_pooled_caveated,S3_shuffled_hwa,781,1.232209,1.235372,-1.078941,1.393976,2.985961,-4.343104,5.410927,...,607,510,781,20260715.0,NaN,NaN,NaN,NaN,0.975775,0.890673
